# Full Data Training - Chunked Approach

**Strategy**: Train on a subset of parquet files at a time, save checkpoints, continue later.

## Models (6 total)
- **CNN**: ConvNet, ResNet
- **Transformer**: SquareTransformer, PieceTransformer  
- **GNN**: GCN, GAT

## Workflow
1. Load 2-3 parquet files → split into train/val/test (80/10/10)
2. Train all 6 models for 15 epochs
3. Save checkpoints
4. Later: load checkpoints, continue with next chunk of files

## 1. Setup

In [1]:
import os
import sys
from pathlib import Path

# Mount Google Drive (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")
    
    possible_paths = [
        '/content/drive/Othercomputers/My MacBook Pro/project',
        '/content/drive/Othercomputers/my macbook pro/project',
        '/content/drive/MyDrive/FYP/project',
    ]
    
    project_path = None
    for path in possible_paths:
        if os.path.exists(path):
            project_path = path
            break
            
    if project_path:
        os.chdir(project_path)
        print(f"Project path: {project_path}")
    else:
        raise FileNotFoundError(f"Project not found.")
        
except ImportError:
    print("Not in Colab. Using current directory.")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

Not in Colab. Using current directory.


In [3]:
!pip install -q python-chess torch torch-geometric pyyaml numpy tqdm pyarrow pandas matplotlib

In [4]:
import gc
import json
import time
from glob import glob
from typing import Optional, Callable
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import chess

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/Users/bhavya/.pyenv/versions/3.14.0/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.10.0
CUDA: False


In [5]:
from src.models.factory import create_model, get_encoder_for_model
from src.chess_env.board_wrapper import UCI_MOVE_TO_INDEX, NUM_MOVES
from src.config import load_config

print(f"UCI moves: {NUM_MOVES}")

UCI moves: 4544


## 2. Configuration

In [6]:
# ============================================================
# EDIT THESE SETTINGS FOR EACH TRAINING RUN
# ============================================================

# Which chunk of files to use (0-indexed)
# Chunk 0: files 0-2, Chunk 1: files 3-5, etc.
CHUNK_ID = 0
FILES_PER_CHUNK = 3

# Resume from checkpoint? Set to True after first run
RESUME_TRAINING = False

# Models to train (comment out ones you've already trained)
MODELS_TO_TRAIN = [
    "convnet",
    "resnet",
    "square_transformer",
    "piece_transformer",
    "gcn",
    "gat",
]

# ============================================================

@dataclass
class Config:
    data_dir: str = "data"
    checkpoint_dir: str = "training_results/full_data"
    
    # Training
    epochs: int = 15
    batch_size: int = 512
    learning_rate: float = 0.001
    weight_decay: float = 0.0001
    
    # Split ratios
    train_ratio: float = 0.80
    val_ratio: float = 0.10
    test_ratio: float = 0.10
    
    # Hardware
    num_workers: int = 2

config = Config()
os.makedirs(config.checkpoint_dir, exist_ok=True)

print(f"Chunk: {CHUNK_ID}, Resume: {RESUME_TRAINING}")
print(f"Models: {MODELS_TO_TRAIN}")

Chunk: 0, Resume: False
Models: ['convnet', 'resnet', 'square_transformer', 'piece_transformer', 'gcn', 'gat']


## 3. Load Data Chunk

In [7]:
# Get parquet files for this chunk
all_files = sorted(glob(os.path.join(config.data_dir, "train-*.parquet")))
print(f"Total parquet files: {len(all_files)}")

start_idx = CHUNK_ID * FILES_PER_CHUNK
end_idx = min(start_idx + FILES_PER_CHUNK, len(all_files))
chunk_files = all_files[start_idx:end_idx]

print(f"\nChunk {CHUNK_ID}: files {start_idx} to {end_idx-1}")
for f in chunk_files:
    print(f"  - {os.path.basename(f)}")

Total parquet files: 17

Chunk 0: files 0 to 2
  - train-00000-of-00017.parquet
  - train-00001-of-00017.parquet
  - train-00002-of-00017.parquet


In [8]:
import pyarrow.parquet as pq

# How many rows to sample per file (None = all rows)
ROWS_PER_FILE = 1_000_000  # Adjust based on RAM (~100K rows ≈ 50-100MB)

print("Loading parquet files (batched)...")
dfs = []

for f in tqdm(chunk_files):
    pf = pq.ParquetFile(f)
    total_rows = pf.metadata.num_rows
    
    if ROWS_PER_FILE and total_rows > ROWS_PER_FILE:
        # Sample rows by reading row groups
        rows_per_group = [pf.metadata.row_group(i).num_rows for i in range(pf.num_row_groups)]
        
        # Read enough row groups to get ROWS_PER_FILE
        collected = 0
        batches = []
        for i, rg_rows in enumerate(rows_per_group):
            if collected >= ROWS_PER_FILE:
                break
            batch = pf.read_row_group(i).to_pandas()
            batches.append(batch)
            collected += rg_rows
        
        df = pd.concat(batches, ignore_index=True)
        # Random sample to exact count
        if len(df) > ROWS_PER_FILE:
            df = df.sample(n=ROWS_PER_FILE, random_state=42)
        del batches
    else:
        df = pd.read_parquet(f)
    
    dfs.append(df)
    print(f"  {os.path.basename(f)}: {len(df):,} rows")

data_df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(f"\nLoaded {len(data_df):,} positions")
print(f"Memory: {data_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Columns: {data_df.columns.tolist()}")
data_df.head(2)

Loading parquet files (batched)...


 33%|███████████████                              | 1/3 [00:00<00:01,  1.63it/s]

  train-00000-of-00017.parquet: 1,000,000 rows


 67%|██████████████████████████████               | 2/3 [00:01<00:00,  1.44it/s]

  train-00001-of-00017.parquet: 1,000,000 rows


100%|█████████████████████████████████████████████| 3/3 [00:02<00:00,  1.48it/s]

  train-00002-of-00017.parquet: 1,000,000 rows

Loaded 3,000,000 positions
Memory: 412.5 MB
Columns: ['fen', 'line', 'depth', 'knodes', 'cp', 'mate']


,fen,line,depth,knodes,cp,mate
0,7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -,f7g7 e6e2 h8d8 e2d2 b7b5 c4b3 g7f6 d1e1 a6a5 a2a3,46,4189972,69.0,NaN
1,7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -,h8d8 d1e1 a6a5 a2a3 c6d7 e6e7 f7f6 e1f2 b7b5 c4b3,46,4189972,163.0,NaN


In [9]:
def parse_best_move(line: str) -> Optional[str]:
    """Extract first move from line (e.g., 'e2e4 d7d5' -> 'e2e4')."""
    if not line or pd.isna(line):
        return None
    moves = str(line).strip().split()
    return moves[0] if moves else None


def cp_to_value(cp: float, mate: Optional[int] = None) -> float:
    """Convert centipawn to value [-1, 1]."""
    if mate is not None and not pd.isna(mate) and mate != 0:
        return 1.0 if mate > 0 else -1.0
    if pd.isna(cp):
        return 0.0
    return float(np.tanh(cp / 400.0))


# Preprocess data: filter invalid rows, add move_idx and value
print("Preprocessing...")

data_df['best_move'] = data_df['line'].apply(parse_best_move)
data_df = data_df.dropna(subset=['best_move', 'fen'])

data_df['move_idx'] = data_df['best_move'].map(UCI_MOVE_TO_INDEX)
data_df = data_df.dropna(subset=['move_idx'])
data_df['move_idx'] = data_df['move_idx'].astype(int)

data_df['value'] = data_df.apply(
    lambda r: cp_to_value(r.get('cp', 0), r.get('mate', None)), axis=1
)

print(f"After filtering: {len(data_df):,} positions")

Preprocessing...
After filtering: 3,000,000 positions


## 4. Dataset & DataLoaders

In [10]:
class ChessDataset(Dataset):
    """In-memory dataset with on-the-fly encoding."""
    
    def __init__(self, df: pd.DataFrame, encoder: Callable):
        self.fens = df['fen'].values
        self.move_idxs = df['move_idx'].values
        self.values = df['value'].values
        self.encoder = encoder
    
    def __len__(self):
        return len(self.fens)
    
    def __getitem__(self, idx):
        fen = self.fens[idx]
        move_idx = self.move_idxs[idx]
        value = self.values[idx]
        
        # Encode board
        board = chess.Board(fen)
        encoded = self.encoder.encode(board)
        
        # Flip value for black
        if not board.turn:
            value = -value
        
        # Policy target (one-hot)
        policy_target = torch.zeros(NUM_MOVES, dtype=torch.float32)
        policy_target[move_idx] = 1.0
        
        return {
            'input': encoded,
            'policy_target': policy_target,
            'value_target': torch.tensor([value], dtype=torch.float32),
        }


def collate_fn(batch):
    """Batch collation."""
    first = batch[0]['input']
    
    if isinstance(first, torch.Tensor):
        inputs = torch.stack([b['input'] for b in batch])
    else:
        inputs = {}
        for key in first.keys():
            vals = [b['input'][key] for b in batch]
            inputs[key] = torch.stack(vals) if torch.is_tensor(vals[0]) else vals
    
    return {
        'input': inputs,
        'policy_target': torch.stack([b['policy_target'] for b in batch]),
        'value_target': torch.stack([b['value_target'] for b in batch]),
    }

In [11]:
# Split data
n = len(data_df)
n_train = int(n * config.train_ratio)
n_val = int(n * config.val_ratio)
n_test = n - n_train - n_val

# Shuffle and split
data_df = data_df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df = data_df.iloc[:n_train]
val_df = data_df.iloc[n_train:n_train+n_val]
test_df = data_df.iloc[n_train+n_val:]

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")

# Free memory - keep only what we need
del data_df
gc.collect()

Train: 2,400,000
Val:   300,000
Test:  300,000


0

## 5. Training Functions

In [12]:
class DualLoss(nn.Module):
    def __init__(self, policy_weight=1.0, value_weight=1.0):
        super().__init__()
        self.pw = policy_weight
        self.vw = value_weight
        self.ce = nn.CrossEntropyLoss()
        self.mse = nn.MSELoss()
    
    def forward(self, out, policy_tgt, value_tgt):
        p_loss = self.ce(out['policy'], policy_tgt)
        v_loss = self.mse(out['value'], value_tgt)
        return {
            'loss': self.pw * p_loss + self.vw * v_loss,
            'policy_loss': p_loss,
            'value_loss': v_loss,
        }


def get_encoder(model_name: str):
    """Get encoder for model type."""
    factory = get_encoder_for_model(model_name)
    if callable(factory):
        try:
            return factory()
        except TypeError:
            return factory
    return factory

In [ ]:
def train_model(
    model_name: str,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    config: Config,
    resume: bool = False,
    chunk_id: int = 0,
) -> dict:
    """Train a single model with continual learning support."""
    print(f"\n{'='*60}")
    print(f"  {model_name.upper()} (Chunk {chunk_id})")
    print(f"{'='*60}")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create model
    base_cfg = load_config("config/gpu_training.yaml")
    base_cfg.model.backbone = model_name
    base_cfg.model.head = "dual"
    
    model = create_model(base_cfg.model).to(device)
    print(f"Parameters: {model.count_parameters():,}")
    
    # Encoder & Datasets
    encoder = get_encoder(model_name)
    train_ds = ChessDataset(train_df, encoder)
    val_ds = ChessDataset(val_df, encoder)
    
    train_loader = DataLoader(
        train_ds, batch_size=config.batch_size, shuffle=True,
        num_workers=config.num_workers, collate_fn=collate_fn, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers, collate_fn=collate_fn, pin_memory=True
    )
    
    loss_fn = DualLoss()
    
    # Resume handling - KEY FIX FOR CONTINUAL LEARNING
    total_epochs_trained = 0
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    best_val_loss = float('inf')
    
    ckpt_path = os.path.join(config.checkpoint_dir, f"{model_name}_latest.pt")
    
    if resume and os.path.exists(ckpt_path):
        print(f"Loading checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        total_epochs_trained = ckpt.get('total_epochs', 0)
        history = ckpt.get('history', history)
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        
        # KEY: Use lower LR for fine-tuning, DON'T load old optimizer state
        fine_tune_lr = config.learning_rate * 0.1  # 10x lower
        actual_epochs = max(5, config.epochs // 2)
        print(f"Fine-tuning: LR={fine_tune_lr:.6f}, epochs={actual_epochs}")
        
        optimizer = AdamW(model.parameters(), lr=fine_tune_lr, weight_decay=config.weight_decay)
    else:
        # Fresh training
        actual_epochs = config.epochs
        optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
        print(f"Fresh training: LR={config.learning_rate}, epochs={actual_epochs}")
    
    scheduler = CosineAnnealingLR(optimizer, T_max=actual_epochs)
    start_time = time.time()
    
    for epoch in range(1, actual_epochs + 1):
        # Train
        model.train()
        train_loss = 0.0
        n_batches = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{actual_epochs}")
        for batch in pbar:
            inputs = batch['input']
            if isinstance(inputs, torch.Tensor):
                inputs = inputs.to(device)
            else:
                inputs = {k: v.to(device) if torch.is_tensor(v) else v for k, v in inputs.items()}
            
            policy_tgt = batch['policy_target'].to(device)
            value_tgt = batch['value_target'].to(device)
            
            optimizer.zero_grad()
            out = model(inputs)
            losses = loss_fn(out, policy_tgt, value_tgt)
            losses['loss'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += losses['loss'].item()
            n_batches += 1
            pbar.set_postfix({'loss': f"{train_loss/n_batches:.4f}"})
        
        scheduler.step()
        avg_train = train_loss / n_batches
        history['train_loss'].append(avg_train)
        
        # Validate
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        n_val = 0
        
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['input']
                if isinstance(inputs, torch.Tensor):
                    inputs = inputs.to(device)
                else:
                    inputs = {k: v.to(device) if torch.is_tensor(v) else v for k, v in inputs.items()}
                
                policy_tgt = batch['policy_target'].to(device)
                value_tgt = batch['value_target'].to(device)
                
                out = model(inputs)
                losses = loss_fn(out, policy_tgt, value_tgt)
                val_loss += losses['loss'].item()
                n_val += 1
                
                pred = out['policy'].argmax(dim=-1)
                target = policy_tgt.argmax(dim=-1)
                correct += (pred == target).sum().item()
                total += pred.size(0)
        
        avg_val = val_loss / n_val
        accuracy = correct / total
        history['val_loss'].append(avg_val)
        history['val_accuracy'].append(accuracy)
        
        print(f"  Train: {avg_train:.4f} | Val: {avg_val:.4f} | Acc: {accuracy:.4f}")
        
        # Save best
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save({
                'model_state_dict': model.state_dict(),
                'val_loss': avg_val,
                'val_accuracy': accuracy,
                'model_name': model_name,
                'chunk_id': chunk_id,
            }, os.path.join(config.checkpoint_dir, f"{model_name}_best.pt"))
            print(f"  ✓ Saved best")
    
    total_epochs_trained += actual_epochs
    
    # Save latest for resuming
    torch.save({
        'model_state_dict': model.state_dict(),
        'total_epochs': total_epochs_trained,
        'history': history,
        'best_val_loss': best_val_loss,
        'last_chunk': chunk_id,
        'model_name': model_name,
    }, ckpt_path)
    
    train_time = time.time() - start_time
    
    # Cleanup
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()
    
    return {
        'model_name': model_name,
        'best_val_loss': best_val_loss,
        'final_accuracy': accuracy,
        'total_epochs': total_epochs_trained,
        'time_minutes': train_time / 60,
        'history': history,
    }


## 6. Train All Models

In [ ]:
results = {}

for model_name in MODELS_TO_TRAIN:
    try:
        result = train_model(
            model_name=model_name,
            train_df=train_df,
            val_df=val_df,
            config=config,
            resume=RESUME_TRAINING,
            chunk_id=CHUNK_ID,
        )
        results[model_name] = result
        
    except Exception as e:
        print(f"\nERROR with {model_name}: {e}")
        import traceback
        traceback.print_exc()
        results[model_name] = {'error': str(e)}
        torch.cuda.empty_cache()
        gc.collect()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)


  CONVNET
Parameters: 4,328,897


Epoch 1/15:   0%|                                      | 0/4688 [00:00<?, ?it/s]/Users/bhavya/.pyenv/versions/3.14.0/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=86, pipe_handle=110)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bhavya/.pyenv/versions/3.14.0/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/bhavya/.pyenv/versions/3.14.0/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: module '__main__' has no attribute 'ChessDataset'


## 7. Results

In [ ]:
# Summary
print(f"\n{'Model':<25} {'Val Loss':<12} {'Accuracy':<12} {'Epochs':<10} {'Time'}")
print("-"*70)

for name, r in results.items():
    if 'error' in r:
        print(f"{name:<25} ERROR")
    else:
        print(f"{name:<25} {r['best_val_loss']:<12.4f} {r['final_accuracy']:<12.4f} {r['total_epochs']:<10} {r['time_minutes']:.1f}m")

# Save summary
summary = {
    'chunk_id': CHUNK_ID,
    'files': [os.path.basename(f) for f in chunk_files],
    'results': {
        k: {kk: vv for kk, vv in v.items() if kk != 'history'}
        for k, v in results.items()
    }
}

with open(os.path.join(config.checkpoint_dir, f"summary_chunk{CHUNK_ID}.json"), 'w') as f:
    json.dump(summary, f, indent=2)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

for i, (name, r) in enumerate(results.items()):
    if 'history' not in r:
        continue
    h = r['history']
    
    axes[0].plot(h['train_loss'], label=name, color=colors[i])
    axes[1].plot(h['val_loss'], label=name, color=colors[i])
    axes[2].plot(h['val_accuracy'], label=name, color=colors[i])

axes[0].set_title('Train Loss')
axes[1].set_title('Val Loss')
axes[2].set_title('Val Accuracy')

for ax in axes:
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.checkpoint_dir, f'curves_chunk{CHUNK_ID}.png'), dpi=150)
plt.show()

## 8. Test Evaluation

In [ ]:
def evaluate_test(model_name: str, test_df: pd.DataFrame, config: Config) -> dict:
    """Evaluate model on test set."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    ckpt_path = os.path.join(config.checkpoint_dir, f"{model_name}_best.pt")
    if not os.path.exists(ckpt_path):
        return {'error': 'No checkpoint'}
    
    base_cfg = load_config("config/gpu_training.yaml")
    base_cfg.model.backbone = model_name
    base_cfg.model.head = "dual"
    
    model = create_model(base_cfg.model).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state_dict'])
    model.eval()
    
    encoder = get_encoder(model_name)
    test_ds = ChessDataset(test_df, encoder)
    test_loader = DataLoader(
        test_ds, batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers, collate_fn=collate_fn
    )
    
    loss_fn = DualLoss()
    test_loss = 0.0
    correct = 0
    total = 0
    n = 0
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Test {model_name}"):
            inputs = batch['input']
            if isinstance(inputs, torch.Tensor):
                inputs = inputs.to(device)
            else:
                inputs = {k: v.to(device) if torch.is_tensor(v) else v for k, v in inputs.items()}
            
            policy_tgt = batch['policy_target'].to(device)
            value_tgt = batch['value_target'].to(device)
            
            out = model(inputs)
            losses = loss_fn(out, policy_tgt, value_tgt)
            test_loss += losses['loss'].item()
            n += 1
            
            pred = out['policy'].argmax(dim=-1)
            target = policy_tgt.argmax(dim=-1)
            correct += (pred == target).sum().item()
            total += pred.size(0)
    
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return {
        'test_loss': test_loss / n,
        'test_accuracy': correct / total,
    }


# Run test evaluation
print("\nTest Set Evaluation")
print("="*50)

test_results = {}
for model_name in MODELS_TO_TRAIN:
    if model_name in results and 'error' not in results[model_name]:
        r = evaluate_test(model_name, test_df, config)
        test_results[model_name] = r
        if 'error' not in r:
            print(f"{model_name:<25} Loss: {r['test_loss']:.4f}  Acc: {r['test_accuracy']:.4f}")

# Save test results
with open(os.path.join(config.checkpoint_dir, f"test_chunk{CHUNK_ID}.json"), 'w') as f:
    json.dump(test_results, f, indent=2)

In [ ]:
# Final ranking
print("\n" + "="*60)
print("FINAL RANKING (by Test Accuracy)")
print("="*60)

ranked = [(n, r['test_accuracy']) for n, r in test_results.items() if 'test_accuracy' in r]
ranked.sort(key=lambda x: x[1], reverse=True)

medals = ['🥇', '🥈', '🥉']
for i, (name, acc) in enumerate(ranked):
    medal = medals[i] if i < 3 else '  '
    print(f"{medal} {i+1}. {name:<25} {acc*100:.2f}%")

---

## Next Steps

To continue training on more data:

1. Change `CHUNK_ID = 1` (or 2, 3, etc.)
2. Set `RESUME_TRAINING = True`
3. Run all cells again

The models will load their saved checkpoints and continue training on the new data chunk.